# Generate/Prepare Writing Elements Zip

Generate or merge writing element glyphs (letters, digits, punctuation).

**Workflow:**
1. **Upload or Generate** writing_elements
2. **Optionally merge** with old zip (keep untrained classes, add new trained classes)
3. **Export FINAL ZIP** ready for training

**Output:** `writing_elements_classifier.zip` with organized glyphs by class.

## Step 1 — Choose: Upload or Generate?

In [ ]:
import os
from google.colab import files

print('Choose an option:')
print('  1. Upload old zip + generate new classes to merge')
print('  2. Generate everything from scratch\n')

response = input('Option (1 or 2): ').strip()

OLD_ZIP_PATH = None
CLASSES_TO_TRAIN = None
GENERATE_SYNTHETIC = True

if response == '1':
    print('\nUpload old writing_elements_classifier.zip:')
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            OLD_ZIP_PATH = filename
    print(f'✅ Old zip: {OLD_ZIP_PATH if OLD_ZIP_PATH else "❌ not found"}')
else:
    print('\n🎲 Will generate everything from scratch...')
    OLD_ZIP_PATH = None
    CLASSES_TO_TRAIN = None

## Step 2 — Configure Selective Training (Optional)

In [ ]:
if OLD_ZIP_PATH:
    print('\nWhich classes to generate? (comma-separated, e.g., "digit,uppercase")')
    print('Options: lowercase, uppercase, digit, punctuation')
    user_input = input().strip()
    if user_input:
        CLASSES_TO_TRAIN = [c.strip() for c in user_input.split(',')]
        print(f'✅ Will generate: {CLASSES_TO_TRAIN}')
    else:
        print('Generating all classes.')
        CLASSES_TO_TRAIN = ['lowercase', 'uppercase', 'digit', 'punctuation']
else:
    CLASSES_TO_TRAIN = ['lowercase', 'uppercase', 'digit', 'punctuation']
    print('\n✅ Will generate all classes from scratch')

# ── AUGMENTATIONS PER CHARACTER ──────────────────────────────────
print('\nAugmentations per character? (default: 75)')
aug_input = input().strip()
AUGMENTATIONS_PER_CHAR = int(aug_input) if aug_input else 75
print(f'✅ {AUGMENTATIONS_PER_CHAR} augmentations per char')

## Step 3 — Generate Synthetic (if needed)

In [ ]:
if GENERATE_SYNTHETIC:
    import string
    import random
    import numpy as np
    from PIL import Image, ImageDraw, ImageFont
    import zipfile
    import shutil
    from collections import defaultdict

    LOWERCASE = string.ascii_lowercase
    UPPERCASE = string.ascii_uppercase
    DIGITS = string.digits
    PUNCTUATION = '.,!?;:\'"()[]{}/-—–_$£€%*~&^><ç#@'

    IMG_SIZE = 32
    FONT_SIZE = 24
    MAX_ROTATION_DEG = 35

    OUTPUT_DIR = '/tmp/writing_elements_glyphs'

    # Find fonts
    FONT_PATHS = []
    font_search_paths = [
        '/usr/share/fonts/truetype/dejavu/',
        '/usr/share/fonts/truetype/liberation/',
        '/usr/share/fonts/truetype/liberation2/',
    ]

    for path in font_search_paths:
        if os.path.exists(path):
            for font_file in os.listdir(path):
                if font_file.endswith('.ttf'):
                    FONT_PATHS.append(os.path.join(path, font_file))

    print(f'Found {len(FONT_PATHS)} fonts')
    print(f'Augmentations per char: {AUGMENTATIONS_PER_CHAR}')

    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    os.makedirs(OUTPUT_DIR)

    for cat in CLASSES_TO_TRAIN:
        os.makedirs(os.path.join(OUTPUT_DIR, cat), exist_ok=True)

    def get_char_category(c):
        if c in LOWERCASE:
            return 'lowercase'
        elif c in UPPERCASE:
            return 'uppercase'
        elif c in DIGITS:
            return 'digit'
        else:
            return 'punctuation'

    def render_char(char, font_path=None, font_size=FONT_SIZE):
        img = Image.new('L', (IMG_SIZE * 2, IMG_SIZE * 2), color=255)
        draw = ImageDraw.Draw(img)
        try:
            if font_path and os.path.exists(font_path):
                font = ImageFont.truetype(font_path, font_size)
            else:
                font = ImageFont.load_default()
        except:
            font = ImageFont.load_default()
        bbox = draw.textbbox((0, 0), char, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        x = (IMG_SIZE * 2 - text_width) // 2
        y = (IMG_SIZE * 2 - text_height) // 2
        draw.text((x, y), char, fill=0, font=font)
        img = img.crop((IMG_SIZE // 2, IMG_SIZE // 2, IMG_SIZE // 2 + IMG_SIZE, IMG_SIZE // 2 + IMG_SIZE))
        return img

    def augment_char(img, aug_idx):
        arr = np.array(img, dtype=np.float32) / 255.0
        brightness = random.uniform(0.8, 1.2)
        arr = np.clip(arr * brightness, 0.0, 1.0)
        pil_img = Image.fromarray((arr * 255).astype(np.uint8))
        pil_img = pil_img.rotate(random.uniform(-MAX_ROTATION_DEG, MAX_ROTATION_DEG),
                                 resample=Image.BICUBIC, fillcolor=255)
        arr = np.array(pil_img, dtype=np.float32) / 255.0
        scale = random.uniform(0.85, 1.15)
        new_size = max(4, int(IMG_SIZE * scale))
        pil_scaled = Image.fromarray((arr * 255).astype(np.uint8)).resize((new_size, new_size), Image.LANCZOS)
        small = np.array(pil_scaled, dtype=np.float32) / 255.0
        canvas = np.ones((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        off = (IMG_SIZE - new_size) // 2
        sy, sx = max(0, off), max(0, off)
        ey = min(sy + small.shape[0], IMG_SIZE)
        ex = min(sx + small.shape[1], IMG_SIZE)
        canvas[sy:ey, sx:ex] = small[:ey - sy, :ex - sx]
        arr = canvas
        if random.random() > 0.5:
            arr = arr[:, ::-1].copy()
        arr += np.random.normal(0, 0.02, arr.shape).astype(np.float32)
        return Image.fromarray((np.clip(arr, 0.0, 1.0) * 255).astype(np.uint8)).convert('L')

    stats = defaultdict(int)
    glyph_id = 0

    print(f'\nGenerating: {CLASSES_TO_TRAIN}')
    
    # Build character set based on selected classes
    chars_to_render = []
    if 'lowercase' in CLASSES_TO_TRAIN:
        chars_to_render.extend(LOWERCASE)
    if 'uppercase' in CLASSES_TO_TRAIN:
        chars_to_render.extend(UPPERCASE)
    if 'digit' in CLASSES_TO_TRAIN:
        chars_to_render.extend(DIGITS)
    if 'punctuation' in CLASSES_TO_TRAIN:
        chars_to_render.extend(PUNCTUATION)

    for char_idx, char in enumerate(chars_to_render):
        category = get_char_category(char)
        if category not in CLASSES_TO_TRAIN:
            continue
        
        font_path = random.choice(FONT_PATHS) if FONT_PATHS else None
        try:
            base_img = render_char(char, font_path=font_path)
        except:
            stats['render_errors'] += 1
            continue
        for aug_idx in range(AUGMENTATIONS_PER_CHAR):
            try:
                aug_img = augment_char(base_img, aug_idx)
                out_path = os.path.join(OUTPUT_DIR, category, f'glyph_{glyph_id:08d}.png')
                aug_img.save(out_path)
                stats[category] += 1
                stats['total'] += 1
                glyph_id += 1
            except:
                stats['aug_errors'] += 1
                continue
        if (char_idx + 1) % 10 == 0:
            pct = 100 * (char_idx + 1) // len(chars_to_render)
            print(f'  {pct:3d}% glyphs: {stats["total"]:,}', flush=True)

    print(f'\nGeneration complete:')
    for cls in CLASSES_TO_TRAIN:
        print(f'  {cls}: {stats[cls]:,}')
    print(f'  Total: {stats["total"]:,}')
else:
    print('Skipping generation.')

## Step 4 — Merge Logic (if applicable)

In [ ]:
import zipfile
import shutil

FINAL_ZIP_PATH = 'writing_elements_classifier.zip'

if OLD_ZIP_PATH and CLASSES_TO_TRAIN:
    print('\nMerging: old untrained classes + new trained classes...')
    
    # Extract old zip
    with zipfile.ZipFile(OLD_ZIP_PATH, 'r') as z:
        z.extractall('/tmp/old_zip')
    
    # Determine new glyphs source
    new_glyphs_source = '/tmp/writing_elements_glyphs'
    
    # Prepare final structure
    final_dir = '/tmp/final_glyphs/glyphs'
    os.makedirs(final_dir, exist_ok=True)
    
    # Copy untrained classes from old
    old_glyphs = '/tmp/old_zip/glyphs'
    for class_name in os.listdir(old_glyphs):
        if class_name not in CLASSES_TO_TRAIN:
            src = os.path.join(old_glyphs, class_name)
            dst = os.path.join(final_dir, class_name)
            if os.path.isdir(src):
                if os.path.exists(dst):
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
                n_glyphs = len([f for f in os.listdir(src) if f.endswith('.png')])
                print(f'  ✅ Kept {class_name}: {n_glyphs} glyphs (from old)')
    
    # Copy trained classes from new
    for class_name in CLASSES_TO_TRAIN:
        src = os.path.join(new_glyphs_source, class_name)
        dst = os.path.join(final_dir, class_name)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            n_glyphs = len([f for f in os.listdir(src) if f.endswith('.png')])
            print(f'  ✅ Added {class_name}: {n_glyphs} glyphs (new)')
    
    print('Merging complete.')
else:
    print('\nNo merge needed. Using all data as-is.')
    final_dir = '/tmp/final_glyphs/glyphs'
    os.makedirs(final_dir, exist_ok=True)
    
    for cat in CLASSES_TO_TRAIN:
        src = os.path.join('/tmp/writing_elements_glyphs', cat)
        dst = os.path.join(final_dir, cat)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)

print('✅ Data prepared.')

## Step 5 — Create and Export Final Zip

In [ ]:
print(f'\nCreating {FINAL_ZIP_PATH}...')

with zipfile.ZipFile(FINAL_ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_list in os.walk('/tmp/final_glyphs'):
        for file in files_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, '/tmp/final_glyphs')
            zf.write(file_path, arcname)

zip_size_mb = os.path.getsize(FINAL_ZIP_PATH) / (1024 * 1024)
print(f'✅ Created {FINAL_ZIP_PATH} ({zip_size_mb:.1f} MB)')

final_glyphs = '/tmp/final_glyphs/glyphs'
print(f'\nFinal structure:')
for class_name in sorted(os.listdir(final_glyphs)):
    class_dir = os.path.join(final_glyphs, class_name)
    if os.path.isdir(class_dir):
        n_glyphs = len([f for f in os.listdir(class_dir) if f.endswith('.png')])
        print(f'  {class_name}: {n_glyphs} glyphs')

## Step 6 — Download

In [ ]:
files.download(FINAL_ZIP_PATH)
print(f'✅ Downloaded {FINAL_ZIP_PATH}')